# KBO 가을야구 예측 전처리 v2 실행본

이 노트북은 KBO 정규시즌 데이터를 가을야구 진출 가능성 분석에 바로 쓸 수 있도록 정리한 실행용 전처리 파일입니다.

실행하면 아래 결과물이 갱신됩니다.

- 원본 데이터 구조 확인
- 2026 가을야구 진출 확률 예측
- 시즌 초반 4월 순위 인사이트
- 대시보드용 CSV 생성
- 발표용 PNG 시각화 생성
- HTML 대시보드 재생성

위에서부터 순서대로 실행하면 됩니다.

In [1]:
from pathlib import Path
import sys
import importlib
import pandas as pd

# 노트북을 어디서 열어도 같은 프로젝트 폴더를 기준으로 실행되게 고정합니다.
PROJECT_DIR = Path(r"C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final")
REPO_DIR = PROJECT_DIR.parent
DATA_DIR = REPO_DIR / "data"
OUT_DIR = PROJECT_DIR / "kbo_outputs"

sys.path.insert(0, str(PROJECT_DIR))

print("프로젝트 폴더:", PROJECT_DIR)
print("데이터 폴더:", DATA_DIR)
print("결과 저장 폴더:", OUT_DIR)

프로젝트 폴더: C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final
데이터 폴더: C:\Users\Admin\Documents\GitHub\Ml_Baseball\data
결과 저장 폴더: C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final\kbo_outputs


## 1. 데이터 폴더 구조 확인

먼저 분석에 필요한 데이터가 실제로 어디에 있는지 확인합니다.

- 과거 원본 데이터: `data/raw/{year}`
- 과거 가공 데이터: `data/processed/{year}`
- 2026 최신 크롤링 데이터: `data/2026`

여기서 파일이 보이면 이후 파이프라인이 정상적으로 데이터를 읽을 수 있습니다.

In [2]:
for path in [DATA_DIR / "raw", DATA_DIR / "processed", DATA_DIR / "2026"]:
    print("\n==", path, "==")
    if not path.exists():
        print("폴더 없음")
        continue
    for item in sorted(path.iterdir())[:20]:
        print(item.name)


== C:\Users\Admin\Documents\GitHub\Ml_Baseball\data\raw ==
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026

== C:\Users\Admin\Documents\GitHub\Ml_Baseball\data\processed ==
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026

== C:\Users\Admin\Documents\GitHub\Ml_Baseball\data\2026 ==
2026_선수_이동_현황.csv
수비_기본기록.csv
주루_기본기록.csv
타자_기본기록.csv
타자_세부기록.csv
투수_기본기록.csv
투수_세부기록.csv
팀_수비_기본기록.csv
팀_수비_세부기록.csv
팀_순위.csv
팀_일자별순위.csv
팀_주루_기본기록.csv
팀_주루_세부기록.csv
팀_타자_기본기록.csv
팀_타자_세부기록.csv
팀_투수_기본기록.csv
팀_투수_세부기록.csv


## 2. 전체 전처리 및 예측 파이프라인 실행

`kbo_postseason_pipeline.run_pipeline()`이 분석용 데이터를 한 번에 만듭니다.

이 셀에서 처리되는 내용은 다음과 같습니다.

- 팀 순위/일자별 순위 불러오기
- 시즌 최종 순위로 가을야구 진출 라벨 생성
- 전년도 팀 성적과 현재 시즌 흐름 결합
- 로지스틱 회귀 모델 학습
- 2026 시즌 각 팀의 진출 확률 계산
- 검증 결과, 4월 순위 인사이트, 모델 데이터 저장

In [3]:
import kbo_postseason_pipeline as pipe
importlib.reload(pipe)

artifacts = pipe.run_pipeline(DATA_DIR, OUT_DIR)
artifacts["summary"]

{'data_dir': 'C:\\Users\\Admin\\Documents\\GitHub\\Ml_Baseball\\data',
 'out_dir': 'C:\\Users\\Admin\\Documents\\GitHub\\Ml_Baseball\\dy_final\\kbo_outputs',
 'training_rows': 4629,
 'model_rows_total': 5190,
 'latest_2026_date': '2026-04-28',
 'predicted_top5': ['KT', 'LG', 'SSG', '삼성', '한화'],
 'april_current_top5_overlap_mean': 3.0,
 'validation_april_ensemble_top5_overlap_mean': 3.0,
 'validation_april_baseline_top5_overlap_mean': 3.0}

## 3. 모델 학습용 데이터 확인

`model_dataset_2023_2026.csv`가 실제 분석에 들어가는 최종 데이터입니다.

팀별로 시즌, 날짜, 순위, 승률, 게임차, 전년도 성적, 가을야구 진출 여부가 합쳐져 있습니다.

In [4]:
model_df = artifacts["model_dataset"]
print("데이터 크기:", model_df.shape)
model_df.head()

데이터 크기: (5190, 61)


,date,rank,team,games,wins,losses,draws,win_rate,games_behind,최근10경기,...,prev_top5_hitter_pa,prev_top5_hitter_xr,prev_top5_hitter_gpa,prev_top5_hitter_isop,prev_top5_pitcher_ip,prev_top5_pitcher_so,prev_top5_pitcher_bb,prev_top5_pitcher_hr,prev_top5_pitcher_era,prev_top5_pitcher_whip
0,2023-04-01,1.0,SSG,1.0,1.0,0.0,0.0,1.0,0.0,1승0무0패,...,2753.0,402.3,0.2796,0.1648,693.000000,551.0,184.0,64.0,3.198,1.184
1,2023-04-01,1.0,키움,1.0,1.0,0.0,0.0,1.0,0.0,1승0무0패,...,2791.0,386.4,0.2692,0.1492,714.666667,569.0,196.0,43.0,3.618,1.268
2,2023-04-01,1.0,KT,1.0,1.0,0.0,0.0,1.0,0.0,1승0무0패,...,2623.0,340.1,0.2636,0.1494,764.000000,624.0,202.0,51.0,3.600,1.288
3,2023-04-01,1.0,NC,1.0,1.0,0.0,0.0,1.0,0.0,1승0무0패,...,2617.0,371.3,0.2780,0.1496,622.333333,585.0,202.0,55.0,3.778,1.310
4,2023-04-01,1.0,두산,1.0,1.0,0.0,0.0,1.0,0.0,1승0무0패,...,2502.0,308.3,0.2550,0.1470,669.666667,536.0,280.0,56.0,4.242,1.488


In [5]:
cols = [
    "season", "date", "team", "rank", "games", "wins", "losses", "win_rate",
    "postseason", "prev_final_rank", "prev_win_rate", "prev_era", "prev_whip"
]
model_df[[c for c in cols if c in model_df.columns]].head(10)

,season,date,team,rank,games,wins,losses,win_rate,postseason,prev_final_rank,prev_win_rate,prev_era,prev_whip
0,2023,2023-04-01,SSG,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.629,3.87,1.29
1,2023,2023-04-01,키움,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.563,3.79,1.34
2,2023,2023-04-01,KT,1.0,1.0,1.0,0.0,1.0,1.0,4.0,0.563,3.51,1.25
3,2023,2023-04-01,NC,1.0,1.0,1.0,0.0,1.0,1.0,6.0,0.475,3.90,1.36
4,2023,2023-04-01,두산,1.0,1.0,1.0,0.0,1.0,1.0,9.0,0.423,4.45,1.48
5,2023,2023-04-01,LG,6.0,1.0,0.0,1.0,0.0,1.0,3.0,0.613,3.33,1.27
6,2023,2023-04-01,KIA,6.0,1.0,0.0,1.0,0.0,0.0,5.0,0.490,4.20,1.42
7,2023,2023-04-01,삼성,6.0,1.0,0.0,1.0,0.0,0.0,7.0,0.465,4.29,1.44
8,2023,2023-04-01,롯데,6.0,1.0,0.0,1.0,0.0,0.0,8.0,0.457,4.45,1.46
9,2023,2023-04-01,한화,6.0,1.0,0.0,1.0,0.0,0.0,10.0,0.324,4.83,1.51


## 4. 2026 가을야구 진출 가능성 예측 결과

예측 결과는 `kbo_outputs/2026_postseason_predictions.csv`에 저장됩니다.

확률은 현재 순위와 승률 흐름, 이전 시즌 팀 전력 지표를 함께 반영해서 계산합니다.

In [6]:
pred = artifacts["predictions"]
pred[[
    "team", "rank", "wins", "losses", "win_rate", "games_behind",
    "model_probability_pct", "standing_probability_pct", "postseason_probability_pct",
    "prediction_label"
]]

,team,rank,wins,losses,win_rate,games_behind,model_probability_pct,standing_probability_pct,postseason_probability_pct,prediction_label
0,KT,1.0,18.0,8.0,0.692,0.0,96.6,88.4,92.1,진출
1,LG,2.0,16.0,9.0,0.640,1.5,95.1,77.9,85.6,진출
2,SSG,3.0,15.0,10.0,0.600,2.5,94.5,70.9,81.5,진출
3,삼성,4.0,13.0,11.0,0.542,4.0,88.8,57.3,71.4,진출
4,한화,7.0,11.0,14.0,0.440,6.5,91.3,36.9,61.4,진출
5,NC,5.0,12.0,13.0,0.480,5.5,59.9,49.6,54.2,미진출
6,KIA,5.0,12.0,13.0,0.480,5.5,56.0,47.9,51.6,미진출
7,두산,8.0,10.0,15.0,0.400,7.5,53.4,28.9,39.9,미진출
8,키움,9.0,10.0,16.0,0.385,8.0,1.7,23.7,13.8,미진출
9,롯데,10.0,8.0,16.0,0.333,9.0,4.0,11.5,8.2,미진출


## 5. 검증 결과와 4월 순위 인사이트

가을야구 예측은 시즌 초반 데이터만으로는 불확실성이 큽니다. 그래서 과거 시즌 기준으로 검증 결과와 4월 순위의 의미를 함께 봅니다.

- `validation_leave_one_season.csv`: 시즌 단위 검증 결과
- `april_rank_insight.csv`: 4월 순위와 최종 성적의 관계

In [7]:
print("시즌 단위 검증 결과")
display(artifacts["validation"])

print("4월 순위 인사이트")
display(artifacts["april"])

시즌 단위 검증 결과


,test_season,slice,rows,team_accuracy,top5_overlap,baseline_current_top5_overlap
0,2023,daily_all_after_10g,1566,0.810983,NaN,NaN
1,2023,april_latest,10,0.600000,3.0,3.0
2,2023,season_latest,10,0.800000,4.0,5.0
3,2024,daily_all_after_10g,1530,0.696732,NaN,NaN
4,2024,april_latest,10,0.500000,3.0,3.0
5,2024,season_latest,10,0.800000,4.0,5.0
6,2025,daily_all_after_10g,1533,0.613829,NaN,NaN
7,2025,april_latest,10,0.700000,3.0,3.0
8,2025,season_latest,10,0.600000,5.0,5.0


4월 순위 인사이트


,season,april_latest_date,current_top5_overlap,current_top5_overlap_rate,rank_final_spearman
0,2023,2023-04-30,3,0.6,0.413376
1,2024,2024-04-30,3,0.6,0.418182
2,2025,2025-04-30,3,0.6,0.684848


## 6. 대시보드용 CSV 생성

HTML 대시보드에서 바로 읽는 CSV 파일을 다시 생성합니다.

생성되는 주요 파일은 다음과 같습니다.

- `2026_가을야구_예측결과.csv`
- `team_master_2022_2026.csv`
- `리그_타격환경.csv`
- `리그_투구환경.csv`

In [8]:
import run_kbo_pipeline_outputs as refresh
importlib.reload(refresh)

refresh.export_dashboard_csvs(DATA_DIR, OUT_DIR)

for p in sorted(PROJECT_DIR.glob("*.csv")):
    print(p.name, p.stat().st_size)

2026_가을야구_예측결과.csv 632
team_master_2022_2026.csv 15025
리그_타격환경.csv 195
리그_투구환경.csv 294


## 7. 발표용 PNG 시각화 생성

리그 흐름, 순위 변화, 투수 지표, 예측 확률을 발표용 이미지로 저장합니다.

생성되는 주요 이미지:

- `01_league_trend.png`
- `02_rank_heatmap.png`
- `03_era_winrate.png`
- `04_playoff_boxplot.png`
- `05_transfer.png`
- `06_feature_importance.png`
- `07_playoff_prob.png`
- `08_apr_vs_final.png`
- `09_cluster_pca.png`

In [9]:
import generate_kbo_visualizations as viz
importlib.reload(viz)

viz.main()

for p in sorted(PROJECT_DIR.glob("0*.png")):
    print(p.name, p.stat().st_size)

C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final\generate_kbo_visualizations.py:169: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="가을야구", y=col, ax=ax, palette={"진출": "#1D9E75", "미진출": "#D85A30"})
C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final\generate_kbo_visualizations.py:169: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="가을야구", y=col, ax=ax, palette={"진출": "#1D9E75", "미진출": "#D85A30"})
C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final\generate_kbo_visualizations.py:169: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.



Generated visualization PNG files in C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final
01_league_trend.png 114338
02_rank_heatmap.png 56019
03_era_winrate.png 65865
04_playoff_boxplot.png 51790
05_transfer.png 12942
06_feature_importance.png 48626
07_playoff_prob.png 46128
08_apr_vs_final.png 40145
09_cluster_pca.png 56713


## 8. HTML 대시보드 재생성

전처리 결과와 시각화 이미지를 반영해서 최종 HTML 대시보드를 다시 만듭니다.

생성 파일:

`kbo_2022_2026_master_dashboard.html`

In [10]:
import generate_kbo_master_dashboard as dash
importlib.reload(dash)

dash.main()

html_path = PROJECT_DIR / "kbo_2022_2026_master_dashboard.html"
print("HTML 위치:", html_path)
print("파일 생성 여부:", html_path.exists())
print("브라우저 주소:", "file:///" + str(html_path).replace("\\", "/"))

C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final\kbo_2022_2026_master_dashboard.html
HTML 위치: C:\Users\Admin\Documents\GitHub\Ml_Baseball\dy_final\kbo_2022_2026_master_dashboard.html
파일 생성 여부: True
브라우저 주소: file:///C:/Users/Admin/Documents/GitHub/Ml_Baseball/dy_final/kbo_2022_2026_master_dashboard.html


## 9. 최종 산출물 정리

이 노트북 실행 후 `dy_final` 폴더에서 확인하면 됩니다.

- 전처리/예측 결과: `dy_final/kbo_outputs`
- 발표용 PNG 자료: `dy_final/01_*.png` ~ `dy_final/09_*.png`
- HTML 대시보드: `dy_final/kbo_2022_2026_master_dashboard.html`
- 자정 자동 갱신 스크립트: `dy_final/run_kbo_midnight_update.ps1`

한글 파일명 노트북이 VSCode에서 빈 상태로 열려 있으면 자동 저장 때문에 다시 비어질 수 있습니다. 그럴 때는 열린 탭을 닫고 `KBO_preprocessing_v2_executable.ipynb`를 기준으로 다시 열면 됩니다.